# NetSentinel — Expert 6: Data Exfiltration (VAE) — Focused DNS Rebuild

**Architecture**: Variational Autoencoder trained on **DNS-discriminative features only** (not all 40+ columns)  
**Dataset**: CIC-Bell-DNS-EXF-2021 (benign DNS for training, exfil DNS for testing)  
**Key fix**: Previous version fed ALL numeric columns to the VAE — the ~30 non-discriminative columns drowned the 8-10 DNS features that carry the actual exfil signal. This version uses only the features where benign ≠ exfil.  

**Datasets attached:**
- `/kaggle/input/datasets/humera11/cicbelldnsexf2021` — PRIMARY
- `/kaggle/input/datasets/madhavmalhotra/unb-cic-iot-dataset` — reference
- `/kaggle/input/datasets/dhoogla/csecicids2018` — benchmark

**Enable GPU**: Settings → Accelerator → GPU T4

In [ ]:
!pip install -q onnxruntime

In [ ]:
import os, glob, gc, json, math, time, warnings
from collections import Counter
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             classification_report, confusion_matrix,
                             f1_score, precision_recall_curve, accuracy_score)
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
print('device:', DEVICE)

EXFIL_ROOT = '/kaggle/input/datasets/humera11/cicbelldnsexf2021'
OUT_DIR = '/kaggle/working/output'
os.makedirs(OUT_DIR, exist_ok=True)

# List all files
all_files = sorted(glob.glob(os.path.join(EXFIL_ROOT, '**', '*.csv'), recursive=True))
print(f'Found {len(all_files)} CSV files:')
for f in all_files:
    sz = os.path.getsize(f) / (1024*1024)
    print(f'  {f}  ({sz:.1f} MB)')

## 1. Load & Label from Directory Structure
CIC-Bell-DNS-EXF-2021 separates classes by **directory** (`.../Benign/` vs `.../Attacks/`) and filename.

In [ ]:
def label_from_path(path):
    """0=benign, 1=exfil. Uses DIRECTORY and FILENAME — never in-file columns."""
    parts = path.lower().replace('\\', '/').split('/')
    for p in parts:
        if 'benign' in p or 'normal' in p or 'legitimate' in p:
            return 0
        if 'attack' in p or 'exfil' in p or 'heavy' in p or 'light' in p:
            return 1
    # filename fallback
    fn = parts[-1]
    if 'benign' in fn or 'normal' in fn:
        return 0
    return 1  # default to attack if ambiguous

def norm_cols(df):
    df = df.copy()
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_'))
    return df

# Load all files, grouped by schema family
all_dfs = []
for f in all_files:
    try:
        df = pd.read_csv(f, low_memory=False)
        df = norm_cols(df)
        label = label_from_path(f)
        label_name = 'BENIGN' if label == 0 else 'EXFIL'
        all_dfs.append((f, df, label))
        print(f'  {label_name:6s} | {df.shape[0]:>8,} rows | {df.shape[1]:>3} cols | {os.path.basename(f)}')
    except Exception as e:
        print(f'  ERR: {os.path.basename(f)}: {e}')

# Group by column signature to find compatible files
from collections import defaultdict
schema_groups = defaultdict(list)
for f, df, label in all_dfs:
    key = tuple(sorted(df.columns))
    schema_groups[key].append((f, df, label))

print(f'\n{len(schema_groups)} distinct schemas found')

# Pick the schema that has BOTH benign and exfil files
best_group = None
for key, items in schema_groups.items():
    labels = set(lbl for _, _, lbl in items)
    n_rows = sum(len(df) for _, df, _ in items)
    has_both = (0 in labels and 1 in labels)
    print(f'  Schema ({len(key)} cols, {n_rows:,} rows): benign={0 in labels}, exfil={1 in labels}')
    if has_both and (best_group is None or n_rows > sum(len(df) for _, df, _ in best_group)):
        best_group = items

assert best_group is not None, 'No schema has both benign and exfil!'

# Concatenate
frames_b, frames_e = [], []
for f, df, label in best_group:
    if label == 0:
        frames_b.append(df)
    else:
        frames_e.append(df)

df_benign = pd.concat(frames_b, ignore_index=True)
df_exfil = pd.concat(frames_e, ignore_index=True)
print(f'\n>>> Selected schema: {len(df_benign.columns)} columns')
print(f'    Benign: {len(df_benign):,} rows')
print(f'    Exfil:  {len(df_exfil):,} rows')
print(f'    Columns: {list(df_benign.columns)}')

## 2. DNS Feature Engineering
The exfil signal lives in the **subdomain string** — length, entropy, digit ratio, special chars. We extract these AND keep the dataset's native numeric DNS features.

In [ ]:
def shannon_entropy(x):
    x = str(x)
    if not x or len(x) == 0:
        return 0.0
    n = len(x)
    return float(-sum((c / n) * math.log2(c / n) for c in Counter(x).values()))

DNS_STR_CANDIDATES = ['subdomain', 'domain', 'query', 'fqdn', 'hostname', 'host',
                      'longest_word', 'sld', 'url', 'name']

def find_dns_string_col(df):
    """Find the best string column that contains the DNS payload."""
    str_cols = [c for c in df.columns if df[c].dtype == object]
    for key in DNS_STR_CANDIDATES:
        for c in str_cols:
            if key in c:
                return c
    return None

def add_dns_features(df):
    """Derive discriminative DNS features from the string column."""
    df = df.copy()
    cand = find_dns_string_col(df)
    if cand is None:
        print('  WARNING: no DNS string column found!')
        return df
    print(f'  Deriving DNS features from: {cand!r}')
    s = df[cand].astype(str)
    L = s.str.len().astype('float32')
    Lp = L + 1.0
    
    df['dns_len'] = L
    df['dns_entropy'] = s.map(shannon_entropy).astype('float32')
    df['dns_digit_ratio'] = (s.str.count(r'[0-9]') / Lp).astype('float32')
    df['dns_upper_ratio'] = (s.str.count(r'[A-Z]') / Lp).astype('float32')
    df['dns_lower_ratio'] = (s.str.count(r'[a-z]') / Lp).astype('float32')
    df['dns_special_ratio'] = (s.str.count(r'[^A-Za-z0-9]') / Lp).astype('float32')
    df['dns_label_count'] = (s.str.count(r'\.') + 1).astype('float32')
    df['dns_longest_token'] = s.str.split(r'[.\-_]').map(
        lambda parts: max((len(p) for p in parts), default=0)).astype('float32')
    df['dns_unique_chars'] = s.map(lambda x: len(set(str(x)))).astype('float32')
    df['dns_max_consecutive'] = s.map(
        lambda x: max((sum(1 for _ in g) for _, g in __import__('itertools').groupby(str(x))), default=0)
    ).astype('float32')
    return df

print('Processing benign...')
df_benign = add_dns_features(df_benign)
print('Processing exfil...')
df_exfil = add_dns_features(df_exfil)

## 3. Feature Selection — ONLY discriminative features
Instead of feeding all 40+ columns, we select features where benign ≠ exfil. This is the critical fix.

In [ ]:
# Drop non-feature columns
DROP_PATTERNS = ['label', 'class', 'attack', 'category', 'type', 'flow_id',
                 'src_ip', 'dst_ip', 'source_ip', 'destination_ip', 'timestamp',
                 'src_port', 'dst_port', 'domain', 'query', 'fqdn', 'hostname',
                 'host', 'sld', 'url', 'name', 'subdomain', 'longest_word', 'id']

def get_numeric_features(df):
    feats = []
    for c in df.columns:
        if any(p in c for p in DROP_PATTERNS):
            continue
        if df[c].dtype == object:
            # try coercing
            conv = pd.to_numeric(df[c], errors='coerce')
            if conv.notna().mean() > 0.9:
                df[c] = conv.astype('float32')
                feats.append(c)
        elif pd.api.types.is_numeric_dtype(df[c]):
            feats.append(c)
    return feats

feats_b = set(get_numeric_features(df_benign))
feats_e = set(get_numeric_features(df_exfil))
common_feats = sorted(feats_b & feats_e)
print(f'Common numeric features: {len(common_feats)}')

# Now check which features actually SEPARATE benign from exfil
# (univariate ROC-AUC on a small sample)
sample_b = df_benign[common_feats].apply(pd.to_numeric, errors='coerce').sample(
    n=min(10000, len(df_benign)), random_state=SEED).values
sample_e = df_exfil[common_feats].apply(pd.to_numeric, errors='coerce').sample(
    n=min(10000, len(df_exfil)), random_state=SEED).values
X_check = np.vstack([sample_b, sample_e])
y_check = np.array([0]*len(sample_b) + [1]*len(sample_e))

feat_auc = []
for i, c in enumerate(common_feats):
    col = X_check[:, i]
    valid = ~np.isnan(col) & ~np.isinf(col)
    if valid.sum() < 100 or np.std(col[valid]) < 1e-12:
        feat_auc.append((c, 0.5, 0.0))
        continue
    auc = roc_auc_score(y_check[valid], col[valid])
    feat_auc.append((c, auc, abs(auc - 0.5)))

feat_auc.sort(key=lambda t: t[2], reverse=True)

print('\n=== Feature discrimination ranking (univariate AUC) ===')
for c, auc, sep in feat_auc[:25]:
    marker = '  ★' if sep > 0.05 else ''
    print(f'  {c:35s}  AUC={auc:.4f}  |AUC-0.5|={sep:.4f}{marker}')

# Keep ONLY features with |AUC-0.5| > 0.03 (weak but nonzero signal)
FEAT_THRESHOLD = 0.03
selected_feats = [c for c, auc, sep in feat_auc if sep > FEAT_THRESHOLD]

# Always include our derived dns_* features if they exist
dns_derived = [c for c in common_feats if c.startswith('dns_')]
for d in dns_derived:
    if d not in selected_feats:
        selected_feats.append(d)

selected_feats = sorted(set(selected_feats))
print(f'\n>>> Selected {len(selected_feats)} discriminative features (out of {len(common_feats)} total)')
print(f'    DNS features: {[f for f in selected_feats if f.startswith("dns_")]}')

assert len(selected_feats) >= 3, f'Only {len(selected_feats)} features — data/label problem'

## 4. Build Clean Matrices + Train/Val/Test Split

In [ ]:
def clean_matrix(df, feats):
    X = df[feats].apply(pd.to_numeric, errors='coerce').astype('float32')
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median())
    return X.values

X_benign = clean_matrix(df_benign, selected_feats)
X_exfil = clean_matrix(df_exfil, selected_feats)

print(f'Benign: {X_benign.shape}')
print(f'Exfil:  {X_exfil.shape}')

# Split benign: 70% train, 15% val (threshold calibration), 15% test
rng = np.random.default_rng(SEED)
idx_b = rng.permutation(len(X_benign))
n_tr = int(0.70 * len(X_benign))
n_vl = int(0.15 * len(X_benign))

Xtr_benign = X_benign[idx_b[:n_tr]]
Xval_benign = X_benign[idx_b[n_tr:n_tr+n_vl]]
Xtest_benign = X_benign[idx_b[n_tr+n_vl:]]

# Split exfil: 50% selection, 50% test
idx_e = rng.permutation(len(X_exfil))
half_e = len(X_exfil) // 2
Xsel_exfil = X_exfil[idx_e[:half_e]]
Xtest_exfil = X_exfil[idx_e[half_e:]]

print(f'\nTrain (benign only):     {Xtr_benign.shape}')
print(f'Val (benign only):       {Xval_benign.shape}')
print(f'Selection (mixed):       benign={Xval_benign.shape[0]}, exfil={Xsel_exfil.shape[0]}')
print(f'Test:                    benign={Xtest_benign.shape[0]}, exfil={Xtest_exfil.shape[0]}')

# Scale (fit on training benign ONLY)
scaler = RobustScaler().fit(Xtr_benign)

def scale(X):
    return np.clip(scaler.transform(X), -10.0, 10.0).astype('float32')

Xtr_s = scale(Xtr_benign)
Xval_s = scale(Xval_benign)
Xsel_b_s = scale(Xval_benign)  # benign part of selection
Xsel_e_s = scale(Xsel_exfil)   # exfil part of selection
Xtest_b_s = scale(Xtest_benign)
Xtest_e_s = scale(Xtest_exfil)

# Build selection and test sets with labels
Xsel_s = np.vstack([Xsel_b_s, Xsel_e_s])
ysel = np.array([0]*len(Xsel_b_s) + [1]*len(Xsel_e_s))

Xte_s = np.vstack([Xtest_b_s, Xtest_e_s])
yte = np.array([0]*len(Xtest_b_s) + [1]*len(Xtest_e_s))

print(f'\nScaled shapes: train={Xtr_s.shape}, val={Xval_s.shape}')
print(f'Selection: {Xsel_s.shape} (benign={int((ysel==0).sum())}, exfil={int(ysel.sum())})')
print(f'Test:      {Xte_s.shape} (benign={int((yte==0).sum())}, exfil={int(yte.sum())})')
print(f'\nBenign fraction in test: {(yte==0).mean():.3f}')
assert (yte==0).sum() > 50, 'Too few benign in test!'
assert (yte==1).sum() > 50, 'Too few exfil in test!'

N_FEATS = Xtr_s.shape[1]
print(f'Input dimension: {N_FEATS}')

## 5. VAE — Sized for the Feature Count

In [ ]:
class VAE(nn.Module):
    def __init__(self, d_in, d_hid=128, d_lat=8):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(d_in, d_hid), nn.LeakyReLU(0.2),
            nn.Linear(d_hid, d_hid//2), nn.LeakyReLU(0.2),
        )
        self.mu = nn.Linear(d_hid//2, d_lat)
        self.lv = nn.Linear(d_hid//2, d_lat)
        self.dec = nn.Sequential(
            nn.Linear(d_lat, d_hid//2), nn.LeakyReLU(0.2),
            nn.Linear(d_hid//2, d_hid), nn.LeakyReLU(0.2),
            nn.Linear(d_hid, d_in),
        )
    
    def forward(self, x):
        h = self.enc(x)
        mu, lv = self.mu(h), self.lv(h)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * lv)
        return self.dec(z), mu, lv
    
    @torch.no_grad()
    def reconstruct(self, x):
        """Deterministic reconstruction (no sampling) for scoring."""
        h = self.enc(x)
        return self.dec(self.mu(h))

def vae_loss(xr, x, mu, lv, beta=1.0):
    rec = nn.functional.mse_loss(xr, x, reduction='none').sum(1)
    kld = -0.5 * torch.sum(1 + lv - mu.pow(2) - lv.exp(), dim=1)
    return (rec + beta * kld).mean()

model = VAE(N_FEATS, d_hid=128, d_lat=8).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)

train_loader = DataLoader(
    TensorDataset(torch.tensor(Xtr_s, dtype=torch.float32)),
    batch_size=512, shuffle=True
)

print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 6. Train on Benign DNS Only (KL Warm-up)

In [ ]:
EPOCHS = 80
PATIENCE = 12
WARMUP = 10

history = {'train_loss': [], 'val_loss': [], 'sel_auc': []}
best_val = float('inf')
best_state = None
wait = 0
start_time = time.time()

@torch.no_grad()
def recon_error(X_scaled):
    """Per-sample reconstruction error (MSE sum)."""
    model.eval()
    errs = []
    for i in range(0, len(X_scaled), 4096):
        xb = torch.tensor(X_scaled[i:i+4096], dtype=torch.float32, device=DEVICE)
        xr = model.reconstruct(xb)
        errs.append(((xr - xb) ** 2).sum(1).cpu().numpy())
    return np.concatenate(errs)

@torch.no_grad()
def calc_val_loss(X_scaled):
    model.eval()
    tot = 0
    for i in range(0, len(X_scaled), 4096):
        xb = torch.tensor(X_scaled[i:i+4096], dtype=torch.float32, device=DEVICE)
        xr, mu, lv = model(xb)
        tot += vae_loss(xr, xb, mu, lv).item() * len(xb)
    return tot / len(X_scaled)

for ep in range(EPOCHS):
    beta = min(1.0, (ep + 1) / WARMUP)
    model.train()
    tot = 0
    for (xb,) in train_loader:
        xb = xb.to(DEVICE)
        xr, mu, lv = model(xb)
        loss = vae_loss(xr, xb, mu, lv, beta=beta)
        opt.zero_grad()
        loss.backward()
        opt.step()
        tot += loss.item() * len(xb)
    
    tl = tot / len(train_loader.dataset)
    vl = calc_val_loss(Xval_s)
    scheduler.step(vl)
    
    # Check selection AUC
    sel_err = recon_error(Xsel_s)
    sel_auc = roc_auc_score(ysel, sel_err)
    # If AUC < 0.5, the score is inverted (benign has HIGHER error) — flip it
    if sel_auc < 0.5:
        sel_auc = 1.0 - sel_auc
    
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['sel_auc'].append(sel_auc)
    
    if (ep + 1) % 5 == 0 or ep == 0:
        print(f'Epoch {ep+1:02d} | TL: {tl:.4f} | VL: {vl:.4f} | β: {beta:.2f} | Sel AUC: {sel_auc:.4f}')
    
    if vl < best_val - 1e-4:
        best_val = vl
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f'Early stop at epoch {ep+1} (best val {best_val:.4f})')
            break

if best_state:
    model.load_state_dict(best_state)

total_time = time.time() - start_time
print(f'\nDone in {total_time/60:.1f} min | Best val loss: {best_val:.4f}')

## 7. Score Calibration + Auto-Flip
If the VAE assigns higher error to benign (inverted), we detect and flip automatically.

In [ ]:
# Get reconstruction errors
val_err = recon_error(Xval_s)       # benign only
test_err = recon_error(Xte_s)       # mixed

# Check if scores need to be flipped
raw_auc = roc_auc_score(yte, test_err)
FLIP = raw_auc < 0.5
if FLIP:
    print(f'Raw AUC = {raw_auc:.4f} < 0.5 — FLIPPING scores (benign has higher recon error)')
    test_err = -test_err
    val_err = -val_err
else:
    print(f'Raw AUC = {raw_auc:.4f} — scores normal (exfil has higher recon error)')

# Threshold at 99th percentile of benign validation error
FPR_BUDGET = 0.01  # 1% false positive rate
thr = float(np.percentile(val_err, 100 * (1 - FPR_BUDGET)))
pred = (test_err > thr).astype(int)

# Metrics
roc = roc_auc_score(yte, test_err)
prc = average_precision_score(yte, test_err)
f1 = f1_score(yte, pred)
acc = accuracy_score(yte, pred)
recall_at_fpr = float((test_err[yte==1] > thr).mean())
realized_fpr = float((test_err[yte==0] > thr).mean())

print(f'\n{"="*55}')
print(f'  EXPERT 6: DATA EXFILTRATION (VAE) — RESULTS')
print(f'{"="*55}')
print(f'  ROC-AUC:          {roc:.4f}')
print(f'  PR-AUC:           {prc:.4f}')
print(f'  F1-Score:         {f1:.4f}')
print(f'  Accuracy:         {acc:.4f}')
print(f'  Recall @ {FPR_BUDGET:.0%} FPR: {recall_at_fpr:.4f}')
print(f'  Realized FPR:     {realized_fpr:.4f}')
print(f'  Score flipped:    {FLIP}')
print(f'{"="*55}')
print()
print(classification_report(yte, pred, target_names=['Benign', 'Exfil']))

## 8. Training Curves + Confusion Matrix + Error Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (0,0) Loss curves
axes[0,0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0,0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0,0].set_title('VAE Training Loss', fontweight='bold')
axes[0,0].legend()
axes[0,0].grid(alpha=0.3)

# (0,1) Selection AUC over epochs
axes[0,1].plot(history['sel_auc'], color='green', linewidth=2)
axes[0,1].set_title('Selection ROC-AUC', fontweight='bold')
axes[0,1].set_ylim(0, 1.02)
axes[0,1].grid(alpha=0.3)
axes[0,1].axhline(y=0.5, ls='--', color='gray', alpha=0.5)

# (1,0) Confusion Matrix
cm = confusion_matrix(yte, pred)
sns.heatmap(cm, annot=True, fmt=',d', cmap='Reds',
            xticklabels=['Benign', 'Exfil'],
            yticklabels=['Benign', 'Exfil'], ax=axes[1,0])
axes[1,0].set_title(f'Confusion Matrix (F1: {f1:.4f})', fontweight='bold')
axes[1,0].set_ylabel('Actual')
axes[1,0].set_xlabel('Predicted')

# (1,1) Error distribution
top = np.percentile(test_err, 99)
bottom = np.percentile(test_err, 1)
bins = np.linspace(bottom, top, 80)
axes[1,1].hist(test_err[yte==0], bins=bins, alpha=0.6, label='Benign', density=True, color='steelblue')
axes[1,1].hist(test_err[yte==1], bins=bins, alpha=0.6, label='Exfil', density=True, color='red')
axes[1,1].axvline(thr, ls='--', color='k', label=f'Threshold (FPR {FPR_BUDGET:.0%})')
axes[1,1].set_title('Anomaly Score Distribution', fontweight='bold')
axes[1,1].legend()

plt.suptitle('Expert 6: Data Exfiltration (VAE Anomaly Detector)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'expert6_graphs.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. ONNX Export + Model Card

In [ ]:
import joblib

# Export deterministic reconstruction path to ONNX
class VAE_Recon(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m
    def forward(self, x):
        return self.m.reconstruct(x)

model_cpu = VAE_Recon(model.cpu()).eval()
dummy = torch.randn(1, N_FEATS)
onnx_path = os.path.join(OUT_DIR, 'expert6_vae.onnx')

torch.onnx.export(
    model_cpu, dummy, onnx_path,
    input_names=['dns_features'], output_names=['reconstruction'],
    dynamic_axes={'dns_features': {0: 'batch'}, 'reconstruction': {0: 'batch'}},
    opset_version=17
)
print(f'ONNX saved: {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)')

# Verify ONNX
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path)
test_in = np.random.randn(1, N_FEATS).astype(np.float32)
onnx_out = sess.run(None, {'dns_features': test_in})
print(f'ONNX output shape: {onnx_out[0].shape}')

# Save scaler and metadata
joblib.dump(scaler, os.path.join(OUT_DIR, 'expert6_scaler.joblib'))

metrics = {
    'model_name': 'NetSentinel Expert 6: Data Exfiltration',
    'model_type': 'Variational Autoencoder (VAE)',
    'version': '1.0.0',
    'task': 'Unsupervised Anomaly Detection (DNS Exfiltration)',
    'roc_auc': float(roc),
    'pr_auc': float(prc),
    'f1': float(f1),
    'accuracy': float(acc),
    'recall_at_fpr': float(recall_at_fpr),
    'fpr_budget': float(FPR_BUDGET),
    'threshold': float(thr),
    'score_flipped': bool(FLIP),
    'training_minutes': float(total_time / 60),
    'input_dim': int(N_FEATS),
    'feature_cols': selected_feats,
    'data_sources': {
        'train_benign': 'CIC-Bell-DNS-EXF-2021 (benign)',
        'test_exfil': 'CIC-Bell-DNS-EXF-2021 (attacks)'
    },
    'mitre_mapping': {
        'T1041': 'Exfiltration Over C2 Channel',
        'T1048': 'Exfiltration Over Alternative Protocol',
        'T1071.004': 'Application Layer Protocol: DNS'
    }
}
with open(os.path.join(OUT_DIR, 'expert6_meta.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

# Model Card
print(f'\n{"="*55}')
print(f'  MODEL CARD: Data Exfiltration VAE (Expert 6)')
print(f'{"="*55}')
print(f'  Arch:      VAE ({N_FEATS}-dim → 128 → 64 → z=8 → 64 → 128 → {N_FEATS}-dim)')
print(f'  Dataset:   CIC-Bell-DNS-EXF-2021')
print(f'  Features:  {N_FEATS} (discriminative DNS features only)')
print(f'  ROC-AUC:   {roc:.4f}')
print(f'  PR-AUC:    {prc:.4f}')
print(f'  F1-Score:  {f1:.4f}')
print(f'  Accuracy:  {acc:.4f}')
print(f'  ONNX:      {onnx_path}')
print(f'  MITRE:     T1041, T1048, T1071.004')
print(f'{"="*55}')

In [ ]:
# ============================================================
# Package all outputs for download
# ============================================================

import zipfile
from IPython.display import FileLink

zip_path = '/kaggle/working/netsentinel_expert6_output.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(OUT_DIR):
        fp = os.path.join(OUT_DIR, f)
        if os.path.isfile(fp):
            zf.write(fp, f)
            print(f'  {f:45s} ({os.path.getsize(fp)/1024:.1f} KB)')

print(f'\nZip: {os.path.getsize(zip_path)/1024/1024:.1f} MB')
FileLink(zip_path)